# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walk-through for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}, Published: {metadata.datePublished}, Version: {metadata.version}")

## 2. Data Overview
Review available record sets and fields by their `@id` values. This helps in referencing the data correctly in downstream steps.

In [ ]:
# List available record sets and their @id's
record_sets = [rs['@id'] for rs in dataset.record_sets]
print("Available Record Set @id's:")
for i, rec_set in enumerate(record_sets):
    print(f"{i+1}. {rec_set}")

# Optionally, show available fields for each record set
for rec_set in dataset.record_sets:
    rec_set_id = rec_set['@id']
    field_ids = [f['@id'] for f in rec_set.get('field', [])]
    print(f"\nRecord Set: {rec_set_id}")
    print("  Field @id's:")
    for f in field_ids:
        print(f"   - {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect all record set IDs, then load the data from each record set.
dataframes = {}
for rec_set in record_sets:
    records = list(dataset.records(record_set=rec_set))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rec_set] = df
        print(f"Loaded {len(df)} records from record set '{rec_set}'. Columns: {df.columns.tolist()}")

# For demonstration, pick the first available (non-empty) record set for downstream EDA
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"\nSelected Record Set for EDA: {selected_record_set_id}")
    print(f"Available columns: {dataframes[selected_record_set_id].columns.tolist()}")
    display(dataframes[selected_record_set_id].head())
else:
    print("No data records could be loaded from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping by key attributes. All operations reference the appropriate `@id` values for dataset entities.

In [ ]:
# If no data loaded, nothing to do.
if len(dataframes) == 0:
    print("No available record set for EDA. Please check the dataset or your network connection.")
else:
    df = dataframes[selected_record_set_id]

    # Display all column names (they correspond to field @id's)
    print(f"Column @id's in selected record set: {df.columns.tolist()}")

    # Attempt to select a numeric field for demo – pick the first column that is numeric
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found in the loaded data.")
    else:
        print(f"Using numeric field for filtering/normalization: {numeric_field_id}")

        # Simple filter: above mean value
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by a categorical field – pick the first non-numeric field
        group_field = None
        for col in df.columns:
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships. Here, we plot the distribution for a selected numeric field and the group means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if len(dataframes) == 0:
    print("No data loaded for visualization.")
else:
    if numeric_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

    if group_field is not None and numeric_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process the FAIR² dataset using the `mlcroissant` library, referencing all data entities by their `@id`. We explored available record sets, extracted data into DataFrames, performed standard EDA and normalization, grouped data by key attributes, and visualized patterns.

For a deeper analysis, consult the dataset schema at the provided Croissant URL and reference specific `@id`s for further, reproducible data science workflows.